# Ondewo s2s client tutorial


First: Importing your clients with these lines:

In [ ]:
from ondewo.nlu.client_config import ClientConfig as NluClientConfig
from ondewo.s2t.client.client_config import ClientConfig as S2tClientConfig
from ondewo.t2s.client.client_config import ClientConfig as T2sClientConfig

from ondewo.nlu.client import Client as NluClient
from ondewo.s2t.client.client import Client as S2tClient
from ondewo.t2s.client.client import Client as T2sClient

Second: Import the grpc auto generated code. This is built automatically from protobuff libraries

In [ ]:
import ondewo.s2t.speech_to_text_pb2 as s2t
import ondewo.t2s.text_to_speech_pb2 as t2s
import ondewo.nlu.agent_pb2 as agent_pb2
import ondewo.nlu.user_pb2 as user_pb2

## Authentication — Keycloak bearer login (D18)

All three clients (NLU, S2T, T2S) now authenticate the same way: **Keycloak headless offline-token auth**.

Fill in the five `KEYCLOAK_*` values in the next cell once and each `ClientConfig` reuses them. On the first RPC the SDK performs a one-time **ROPC** grant (`scope=offline_access`) against the realm's **public** Keycloak client (`ondewo-nlu-cai-sdk-public`, no client secret), then attaches the resulting token as `Authorization: Bearer <jwt>` metadata on every call and auto-refreshes it in the background.

The legacy `http_token` (HTTP-Basic) and `users.login()` / Login-RPC (`cai-token`) flows have been **removed** — authentication is bearer-only.

> `grpc_cert` is orthogonal to authentication: it still controls the **TLS transport** of the gRPC channel (leave it set for a secure channel). The bearer token rides on top of that channel.

The username field name differs per client: **NLU uses `user_name`**, while **S2T and T2S use `username`**.

In [ ]:
# --- Keycloak headless offline-token auth (D18) -------------------------------
# Fill these in once; every client below reuses them.
# Leave the realm / client_id defaults unless your platform overrides them.
KEYCLOAK_URL = ""                                # e.g. "https://keycloak.ondewo.com/auth" (base URL before /realms/<realm>)
KEYCLOAK_REALM = "ondewo-ccai-platform"
KEYCLOAK_CLIENT_ID = "ondewo-nlu-cai-sdk-public"
KEYCLOAK_USERNAME = ""                           # project technical user or admin@ondewo.com
KEYCLOAK_PASSWORD = ""

nlu_config: NluClientConfig = NluClientConfig(
    host = "",
    port = "",
    grpc_cert = "",
    keycloak_url = KEYCLOAK_URL,
    realm = KEYCLOAK_REALM,
    client_id = KEYCLOAK_CLIENT_ID,
    user_name = KEYCLOAK_USERNAME,   # NLU uses `user_name` for the Keycloak path
    password = KEYCLOAK_PASSWORD,
)
s2t_config: S2tClientConfig = S2tClientConfig(
    host = "",
    port = "",
    grpc_cert = "",
    keycloak_url = KEYCLOAK_URL,
    realm = KEYCLOAK_REALM,
    client_id = KEYCLOAK_CLIENT_ID,
    username = KEYCLOAK_USERNAME,     # S2T uses `username`
    password = KEYCLOAK_PASSWORD,
)
t2s_config: T2sClientConfig = T2sClientConfig(
    host = "",
    port = "",
    grpc_cert = "",
    keycloak_url = KEYCLOAK_URL,
    realm = KEYCLOAK_REALM,
    client_id = KEYCLOAK_CLIENT_ID,
    username = KEYCLOAK_USERNAME,     # T2S uses `username`
    password = KEYCLOAK_PASSWORD,
)


## s2t, t2s and nlu Clients
Now you have all the clients 

In [ ]:
nlu_client = NluClient(config=nlu_config, use_secure_channel=True)

In [ ]:
s2t_client = S2tClient(config=s2t_config, use_secure_channel=True)

In [ ]:
t2s_client = T2sClient(config=t2s_config, use_secure_channel=True)

### Sending the Keycloak bearer on the S2T / T2S calls

The **NLU** client auto-attaches the bearer to every RPC (`metadata=self.metadata` on each service call), so its calls further down stay high-level.

The **S2T** and **T2S** high-level wrappers (`transcribe_file`, `synthesize`, `list_*_pipelines`, …) take **no** `metadata` argument and there is no interceptor, so they do **not** forward the token — calling them directly would fail `UNAUTHENTICATED`. The canonical pattern is to build one shared token provider per config (the ROPC offline-token login runs once, then auto-refreshes) and issue the call through the **low-level stub**:

`service.stub.<Rpc>(request, metadata=provider.bearer_metadata())`

In [ ]:
# get some helper functions
%run Jupyter_demo_helper.ipynb

#### Example 1 s2t 

In [ ]:
AUDIO_FILE: str = "s2t_examples/audiofiles/sample_1.wav"
language = 'en'

transcribe_response = s2t_ex_1(AUDIO_FILE, language)

In [ ]:
print(transcribe_response.transcriptions)

s2t_pipelines = s2t_client.services.speech_to_text.list_s2t_pipelines(request=s2t.ListS2tPipelinesRequest())
print(f"Speech to text pipelines: {[pipeline.id for pipeline in s2t_pipelines.pipeline_configs]}")

print(f"Speech to text domains: { set([pipeline.description.domain for pipeline in s2t_pipelines.pipeline_configs])}")

print(f"Speech to text languages: { set([pipeline.description.language for pipeline in s2t_pipelines.pipeline_configs])}")


####  Example 2 s2t

In [ ]:
AUDIO_FILE: str = "s2t_examples/audiofiles/sample_2.wav"
language = 'de'

response_gen = s2t_ex_2(AUDIO_FILE, language)

In [ ]:
for i, response_chunk in enumerate(response_gen):
    print(response_chunk.transcriptions)

In [ ]:
s2t_pipelines = s2t_client.services.speech_to_text.list_s2t_pipelines(request=s2t.ListS2tPipelinesRequest())
print(f"Speech to text pipelines: {[pipeline.id for pipeline in s2t_pipelines.pipeline_configs]}")

print(f"Speech to text domains: { set([pipeline.description.domain for pipeline in s2t_pipelines.pipeline_configs])}")

print(f"Speech to text languages: { set([pipeline.description.language for pipeline in s2t_pipelines.pipeline_configs])}")


#### Example 3 s2t : live stream

In [ ]:
def live_speech(pipeline_id,session_id = str(uuid.uuid4()),save_to_disk = False,streamer_name = "pyaudio"):
    live_speech_helper(pipeline_id,session_id,save_to_disk,streamer_name)

In [ ]:
live_speech('default_german')

## t2s client


#### Example t2s

In [ ]:
t2s_pipelines = t2s_client.services.text_to_speech.list_t2s_pipelines(request=t2s.ListT2sPipelinesRequest())
print(f"Text to speech pipelines: {[pipeline.id for pipeline in t2s_pipelines.pipelines]}")

print(f"Text to speech domains: {set([pipeline.description.domain for pipeline in t2s_pipelines.pipelines])}")
print(f"Text to speech languages: {set([pipeline.description.language for pipeline in t2s_pipelines.pipelines])}")

In [ ]:
voice_num = 1
language = "en"
length_scale = 10
say(t2s_client, "Gruess dich Ich bin Gabriel", length_scale, language, voice_num)

## NLU client 

In [ ]:
# get some helper functions
%run Jupyter_demo_helper.ipynb

In [ ]:
project_id =  "e4dc4aed-99bd-4f4d-9355-a38a574d3370"
get_agent_by_id(project_id)
#session_id can be passed too
session = create_session_nlu(project_id)
get_agent_by_name(session)

In [ ]:
# Enter "end convo" to quit at any time
nlu_convo()

## s2s example

In [ ]:
# TODO 